# LeetCode 2250 — Count Number of Rectangles Containing Each Point bằng Python

Link đề: https://leetcode.com/problems/count-number-of-rectangles-containing-each-point/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Binary Search, Binary Indexed Tree, Sorting

Skeleton LeetCode:

```python
class Solution:
    def countRectangles(self, rectangles: List[List[int]], points: List[List[int]]) -> List[int]:
        ...
```

## 1. Mục tiêu học

1. Đọc **ràng buộc lệch**: chiều dài tới 10⁹ nhưng chiều cao chỉ ≤ 100 ⇒ có thể **duyệt theo chiều cao** và **tìm nhị phân theo chiều dài**.
2. Thành thạo `bisect_left` để đếm "số phần tử ≥ x" trong danh sách đã sắp xếp.
3. Biết kỹ thuật **xử lý ngoại tuyến** (offline): sắp xếp cả truy vấn và dữ liệu, quét một chiều, duy trì cấu trúc cho chiều còn lại (mảng đếm / Fenwick tree / `SortedList`).
4. Bài toán "đếm điểm trội" (dominance counting) 2 chiều: hình chữ nhật chứa điểm ⇔ `l ≥ x` **và** `h ≥ y`.
5. Đối chiếu nhiều cách với vét cạn; giữ đúng **thứ tự đáp án** theo thứ tự truy vấn ban đầu khi xử lý ngoại tuyến.

## 2. Đề bài & ví dụ

Cho `rectangles[i] = [l_i, h_i]`: hình chữ nhật thứ `i` có góc dưới-trái `(0, 0)` và góc trên-phải `(l_i, h_i)`. Cho `points[j] = [x_j, y_j]`.

Trả về mảng `count` với `count[j]` = số hình chữ nhật **chứa** điểm `j`. Hình chữ nhật chứa điểm nếu `0 ≤ x_j ≤ l_i` và `0 ≤ y_j ≤ h_i` (điểm trên cạnh cũng tính).

**Ví dụ 1:** `rectangles = [[1,2],[2,3],[2,5]]`, `points = [[2,1],[1,4]]` → `[2, 1]`.

**Ví dụ 2:** `rectangles = [[1,1],[2,2],[3,3]]`, `points = [[1,3],[1,1]]` → `[1, 3]`.

**Ràng buộc:** `1 ≤ len(rectangles), len(points) ≤ 5·10⁴`, `1 ≤ l_i, x_j ≤ 10⁹`, `1 ≤ h_i, y_j ≤ 100`; các hình chữ nhật phân biệt, các điểm phân biệt.

In [1]:
from typing import List
from bisect import bisect_left, insort
from collections import defaultdict
import random, time

examples = [
    ([[1, 2], [2, 3], [2, 5]], [[2, 1], [1, 4]], [2, 1]),
    ([[1, 1], [2, 2], [3, 3]], [[1, 3], [1, 1]], [1, 3]),
]
for rects, pts, exp in examples:
    for x, y in pts:
        who = [r for r in rects if x <= r[0] and y <= r[1]]
        print(f"điểm {(x, y)} nằm trong {who}")
    print("kỳ vọng", exp)

điểm (2, 1) nằm trong [[2, 3], [2, 5]]
điểm (1, 4) nằm trong [[2, 5]]
kỳ vọng [2, 1]
điểm (1, 3) nằm trong [[3, 3]]
điểm (1, 1) nằm trong [[1, 1], [2, 2], [3, 3]]
kỳ vọng [1, 3]


## 3. Kiến thức nền

### 3.1 Điều kiện chứa = "trội cả hai chiều"

Vì mọi hình chữ nhật đều bắt đầu ở gốc `(0, 0)` và điểm có tọa độ dương, điều kiện chỉ còn `x ≤ l` **và** `y ≤ h`: góc trên-phải của hình chữ nhật **trội** (dominate) điểm.

### 3.2 `bisect_left` — đếm số phần tử ≥ x

Trong danh sách tăng dần `a`, `bisect_left(a, x)` là số phần tử `< x` ⇒ số phần tử `≥ x` là `len(a) − bisect_left(a, x)`.

In [2]:
a = [1, 2, 2, 5, 9]
for x in [0, 2, 3, 9, 10]:
    print(f"x={x}: số phần tử >= x = {len(a) - bisect_left(a, x)}")

x=0: số phần tử >= x = 5
x=2: số phần tử >= x = 4
x=3: số phần tử >= x = 2
x=9: số phần tử >= x = 1
x=10: số phần tử >= x = 0


### 3.3 Nhóm theo khóa bằng `defaultdict(list)` rồi sắp xếp từng nhóm

In [3]:
by_h = defaultdict(list)
for l, h in [[1, 2], [2, 3], [2, 5], [7, 3]]:
    by_h[h].append(l)
for h in by_h: by_h[h].sort()
print(dict(by_h))

{2: [1], 3: [2, 7], 5: [2]}


### 3.4 Cây Fenwick (Binary Indexed Tree) — cộng điểm, hỏi tổng tiền tố trong O(log N)

Dùng khi cần "thêm 1 vào vị trí `h`" và "đếm số phần tử có vị trí ≥ y" xen kẽ nhau.

In [4]:
class Fenwick:
    def __init__(self, n):
        self.n = n; self.t = [0] * (n + 1)
    def add(self, i, v=1):              # i từ 1..n
        while i <= self.n:
            self.t[i] += v; i += i & -i
    def prefix(self, i):                # tổng [1..i]
        s = 0
        while i > 0:
            s += self.t[i]; i -= i & -i
        return s

fw = Fenwick(10)
for h in [2, 3, 5, 3]: fw.add(h)
print("số phần tử <= 3:", fw.prefix(3), "| số phần tử >= 3:", fw.prefix(10) - fw.prefix(2))

số phần tử <= 3: 3 | số phần tử >= 3: 3


### 3.5 `SortedList` (thư viện `sortedcontainers`, LeetCode hỗ trợ)

Danh sách luôn sắp xếp, `add` O(log n), `bisect_left` O(log n). Thay thế cho `insort` (O(n) vì phải dịch phần tử).

In [5]:
from sortedcontainers import SortedList
sl = SortedList([5, 1, 3]); sl.add(2)
print(sl, "số phần tử >= 3:", len(sl) - sl.bisect_left(3))

SortedList([1, 2, 3, 5]) số phần tử >= 3: 2


## 4. Ý tưởng nền tảng / trực giác

**Ràng buộc lệch là chìa khóa.** Chiều dài có thể tới 10⁹ (không thể đánh chỉ số mảng), nhưng chiều cao ≤ 100 — chỉ có tối đa 100 giá trị khác nhau.

Với điểm `(x, y)`: đáp án = Σ trên mọi chiều cao `h ≥ y` của (số hình chữ nhật có chiều cao `h` **và** chiều dài `≥ x`). Mỗi số hạng trong tổng: tìm nhị phân trong danh sách chiều dài đã sắp xếp của nhóm `h`.

Tính tay ví dụ 1. Nhóm theo chiều cao: `h=2: [1]`, `h=3: [2]`, `h=5: [2]`.
- Điểm `(2, 1)`: các nhóm `h ≥ 1`: `h=2` có `0` chiều dài ≥ 2; `h=3` có `1`; `h=5` có `1` ⇒ **2**.
- Điểm `(1, 4)`: chỉ nhóm `h=5` (`≥ 4`): `1` chiều dài ≥ 1 ⇒ **1**.

**Góc nhìn khác (ngoại tuyến):** sắp xếp điểm theo `x` **giảm dần**. Khi xét điểm có `x` nhỏ hơn, tập hình chữ nhật có `l ≥ x` chỉ **lớn thêm** ⇒ thêm dần các hình chữ nhật (theo `l` giảm dần) vào một cấu trúc đếm theo chiều cao, rồi hỏi "bao nhiêu cái có `h ≥ y`".

In [6]:
rects, pts, _ = examples[0]
by_h = defaultdict(list)
for l, h in rects: by_h[h].append(l)
for h in by_h: by_h[h].sort()
for x, y in pts:
    parts = {h: len(ls) - bisect_left(ls, x) for h, ls in sorted(by_h.items()) if h >= y}
    print(f"điểm {(x, y)}: theo nhóm chiều cao {parts} -> tổng {sum(parts.values())}")

điểm (2, 1): theo nhóm chiều cao {2: 0, 3: 1, 5: 1} -> tổng 2
điểm (1, 4): theo nhóm chiều cao {5: 1} -> tổng 1


## 5. Cách 1 — Brute Force / baseline: so mọi cặp (điểm, hình chữ nhật)

In [7]:
def brute(rectangles, points):
    return [sum(1 for l, h in rectangles if x <= l and y <= h) for x, y in points]

for r, p, exp in examples:
    got = brute(r, p)
    print(got, "OK" if got == exp else "SAI")

[2, 1] OK
[1, 3] OK


**Độ phức tạp:** O(R · P) = `5·10⁴ · 5·10⁴ = 2,5·10⁹` ⇒ TLE.

**Nút thắt:** với mỗi điểm ta xét lại mọi hình chữ nhật, không tận dụng thứ tự. Nếu các chiều dài được **sắp xếp**, "đếm số chiều dài ≥ x" chỉ tốn O(log R) — nhưng còn điều kiện chiều cao; tách theo chiều cao (≤ 100 nhóm) giải quyết được.

In [8]:
rng = random.Random(1)
R = [[rng.randint(1, 10**9), rng.randint(1, 100)] for _ in range(3000)]
P = [[rng.randint(1, 10**9), rng.randint(1, 100)] for _ in range(3000)]
t = time.perf_counter(); brute(R, P); print(f"brute 3000 x 3000: {(time.perf_counter()-t)*1000:.0f} ms (đề thật lớn hơn 280 lần)")

brute 3000 x 3000: 257 ms (đề thật lớn hơn 280 lần)


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 — nhóm theo chiều cao + tìm nhị phân:** `count(x, y) = Σ_{h=y}^{100} (|L_h| − bisect_left(L_h, x))`, với `L_h` là danh sách chiều dài (đã sắp) của các hình chữ nhật có chiều cao đúng `h`. Mỗi điểm tốn ≤ 100 lần tìm nhị phân.

**Insight 2 — gộp tiền tố theo chiều cao:** nếu thay `L_h` bằng `G_h` = danh sách chiều dài của mọi hình chữ nhật có chiều cao **≥ h**, mỗi điểm chỉ cần **một** lần tìm nhị phân trong `G_y`. Đổi lại tốn bộ nhớ: tổng kích thước `Σ |G_h|` có thể tới `100 · R = 5·10⁶`.

**Insight 3 — ngoại tuyến theo x:** sắp điểm theo `x` giảm dần, hình chữ nhật theo `l` giảm dần; con trỏ thêm mọi hình chữ nhật có `l ≥ x` vào mảng đếm theo chiều cao `cnt[h]`; đáp án = `Σ_{h ≥ y} cnt[h]` (≤ 100 phép cộng, hoặc O(log 100) với Fenwick).

**Insight 4 — ngoại tuyến theo y:** sắp điểm theo `y` giảm dần; thêm chiều dài của mọi hình chữ nhật có `h ≥ y` vào `SortedList`; đáp án = số phần tử ≥ `x`.

**Quy tắc:** khi xử lý ngoại tuyến, **luôn giữ chỉ số gốc** của truy vấn để ghi đáp án đúng vị trí.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Nhóm theo chiều cao + `bisect` (cách doocs)

In [9]:
def group_bisect(rectangles, points):
    by_h = defaultdict(list)
    for l, h in rectangles:
        by_h[h].append(l)
    for h in by_h:
        by_h[h].sort()
    ans = []
    for x, y in points:
        c = 0
        for h in range(y, 101):
            ls = by_h.get(h)
            if ls:
                c += len(ls) - bisect_left(ls, x)
        ans.append(c)
    return ans

for r, p, exp in examples:
    got = group_bisect(r, p)
    print(got, "OK" if got == exp else "SAI")

[2, 1] OK
[1, 3] OK


**Độ phức tạp cách 2:** sắp xếp O(R log R); mỗi điểm O(100 · log R) ⇒ tổng O(R log R + 100 · P · log R) ≈ 8·10⁷ phép cơ bản worst — nhưng phần lớn là `bisect` viết bằng C nên chạy được.

### 7.2 Cách 3 — Danh sách gộp "chiều cao ≥ h" (một lần tìm nhị phân mỗi điểm)

In [10]:
def merged_suffix(rectangles, points):
    by_h = defaultdict(list)
    for l, h in rectangles: by_h[h].append(l)
    G = [[] for _ in range(102)]
    acc = []
    for h in range(100, 0, -1):
        acc = sorted(acc + by_h.get(h, []))        # mọi chiều dài có chiều cao >= h
        G[h] = acc
    return [len(G[y]) - bisect_left(G[y], x) for x, y in points]

for r, p, exp in examples:
    got = merged_suffix(r, p)
    print(got, "OK" if got == exp else "SAI")

[2, 1] OK
[1, 3] OK


**Độ phức tạp cách 3:** tiền xử lý O(100 · R log R) và bộ nhớ O(100 · R) worst; mỗi truy vấn O(log R). Phù hợp khi **rất nhiều** truy vấn và bộ nhớ dư dả. (Có thể gộp kiểu merge thay vì `sorted` để còn O(100 · R).)

### 7.3 Cách 4 — Ngoại tuyến theo x giảm dần + mảng đếm theo chiều cao

In [11]:
def offline_by_x(rectangles, points):
    rs = sorted(rectangles, key=lambda r: -r[0])
    order = sorted(range(len(points)), key=lambda j: -points[j][0])
    cnt = [0] * 102; ans = [0] * len(points); k = 0
    for j in order:
        x, y = points[j]
        while k < len(rs) and rs[k][0] >= x:
            cnt[rs[k][1]] += 1; k += 1
        ans[j] = sum(cnt[y:101])
    return ans

for r, p, exp in examples:
    got = offline_by_x(r, p)
    print(got, "OK" if got == exp else "SAI")

[2, 1] OK
[1, 3] OK


Thay `sum(cnt[y:101])` bằng Fenwick để có O(log 100) mỗi truy vấn:

In [12]:
def offline_fenwick(rectangles, points):
    rs = sorted(rectangles, key=lambda r: -r[0])
    order = sorted(range(len(points)), key=lambda j: -points[j][0])
    fw = Fenwick(100); ans = [0] * len(points); k = 0; total = 0
    for j in order:
        x, y = points[j]
        while k < len(rs) and rs[k][0] >= x:
            fw.add(rs[k][1]); k += 1; total += 1
        ans[j] = total - fw.prefix(y - 1)          # số chiều cao >= y
    return ans

for r, p, exp in examples:
    got = offline_fenwick(r, p)
    print(got, "OK" if got == exp else "SAI")

[2, 1] OK
[1, 3] OK


**Độ phức tạp cách 4:** O(R log R + P log P) sắp xếp + O(R + P · log 100) quét. Bộ nhớ O(R + P).

### 7.4 Cách 5 — Ngoại tuyến theo y giảm dần + `SortedList` chiều dài

In [13]:
def offline_by_y(rectangles, points):
    rs = sorted(rectangles, key=lambda r: -r[1])
    order = sorted(range(len(points)), key=lambda j: -points[j][1])
    sl = SortedList(); ans = [0] * len(points); k = 0
    for j in order:
        x, y = points[j]
        while k < len(rs) and rs[k][1] >= y:
            sl.add(rs[k][0]); k += 1
        ans[j] = len(sl) - sl.bisect_left(x)
    return ans

for r, p, exp in examples:
    got = offline_by_y(r, p)
    print(got, "OK" if got == exp else "SAI")

[2, 1] OK
[1, 3] OK


**Độ phức tạp cách 5:** O((R + P) log R). Không phụ thuộc vào việc chiều cao nhỏ — dùng được cả khi **cả hai** chiều tới 10⁹ (bài "dominance counting" tổng quát).

## 8. TRACE từng bước

### 8.1 Trace cách 4 (ngoại tuyến theo x) trên ví dụ 2

`rectangles = [[1,1],[2,2],[3,3]]` sắp theo `l` giảm: `[3,3], [2,2], [1,1]`. Điểm theo `x` giảm: cả hai điểm có `x = 1`.

In [14]:
def trace_offline_x(rectangles, points):
    rs = sorted(rectangles, key=lambda r: -r[0])
    order = sorted(range(len(points)), key=lambda j: -points[j][0])
    cnt = [0] * 102; ans = [0] * len(points); k = 0
    print("hình chữ nhật theo l giảm:", rs)
    for j in order:
        x, y = points[j]
        added = []
        while k < len(rs) and rs[k][0] >= x:
            cnt[rs[k][1]] += 1; added.append(rs[k]); k += 1
        ans[j] = sum(cnt[y:101])
        nonzero = {h: c for h, c in enumerate(cnt) if c}
        print(f"điểm #{j} {(x, y)}: thêm {added}, đếm theo chiều cao {nonzero}, số có h >= {y}: {ans[j]}")
    return ans

print("đáp án:", trace_offline_x(*examples[1][:2]))

hình chữ nhật theo l giảm: [[3, 3], [2, 2], [1, 1]]
điểm #0 (1, 3): thêm [[3, 3], [2, 2], [1, 1]], đếm theo chiều cao {1: 1, 2: 1, 3: 1}, số có h >= 3: 1
điểm #1 (1, 1): thêm [], đếm theo chiều cao {1: 1, 2: 1, 3: 1}, số có h >= 1: 3
đáp án: [1, 3]


### 8.2 Trace cách 2 trên một bộ dữ liệu có nhiều nhóm chiều cao

In [15]:
rects = [[4, 1], [6, 3], [2, 3], [9, 2], [5, 5], [7, 5], [3, 4]]
pts = [[3, 2], [6, 4], [1, 1], [8, 5]]
by_h = defaultdict(list)
for l, h in rects: by_h[h].append(l)
for h in by_h: by_h[h].sort()
print("nhóm:", dict(sorted(by_h.items())))
for x, y in pts:
    parts = []
    for h in range(y, 101):
        if h in by_h:
            c = len(by_h[h]) - bisect_left(by_h[h], x); parts.append(f"h={h}:{c}")
    print(f"điểm {(x, y)} -> {', '.join(parts)} -> tổng {group_bisect(rects, [[x, y]])[0]}")
print("brute:", brute(rects, pts))

nhóm: {1: [4], 2: [9], 3: [2, 6], 4: [3], 5: [5, 7]}
điểm (3, 2) -> h=2:1, h=3:1, h=4:1, h=5:2 -> tổng 5
điểm (6, 4) -> h=4:0, h=5:1 -> tổng 1
điểm (1, 1) -> h=1:1, h=2:1, h=3:2, h=4:1, h=5:2 -> tổng 7
điểm (8, 5) -> h=5:0 -> tổng 0
brute: [5, 1, 7, 0]


## 9. Code nộp LeetCode

In [16]:
from typing import List
from bisect import bisect_left
from collections import defaultdict

class Solution:
    def countRectangles(self, rectangles: List[List[int]], points: List[List[int]]) -> List[int]:
        by_h = defaultdict(list)                 # chiều cao (<= 100) -> các chiều dài
        for l, h in rectangles:
            by_h[h].append(l)
        heights = sorted(by_h)                   # chỉ các chiều cao thực sự xuất hiện
        for h in heights:
            by_h[h].sort()
        ans = []
        for x, y in points:
            c = 0
            for h in heights[bisect_left(heights, y):]:     # mọi chiều cao >= y
                ls = by_h[h]
                c += len(ls) - bisect_left(ls, x)           # số chiều dài >= x
            ans.append(c)
        return ans

In [17]:
sol = Solution()
for r, p, exp in examples:
    got = sol.countRectangles(r, p)
    print(got, "| kỳ vọng", exp, "| OK" if got == exp else "| SAI")

[2, 1] | kỳ vọng [2, 1] | OK
[1, 3] | kỳ vọng [1, 3] | OK


Mẹo nhỏ trong bản nộp: chỉ duyệt các chiều cao **thực sự xuất hiện** và bắt đầu từ chiều cao đầu tiên `≥ y` (tìm bằng `bisect` trên danh sách chiều cao) — bỏ qua các nhóm rỗng.

## 10. Vì sao thuật toán đúng

**Cách 2.** Tập hình chữ nhật chứa `(x, y)` là `{i : l_i ≥ x, h_i ≥ y}`. Phân hoạch theo giá trị `h_i`: `Σ_{h ≥ y} |{i : h_i = h, l_i ≥ x}|`. Với mỗi `h`, danh sách `L_h` đã sắp tăng dần nên số phần tử `≥ x` đúng bằng `|L_h| − bisect_left(L_h, x)` (bisect_left trả số phần tử `< x`). Các nhóm rời nhau ⇒ tổng không trùng. ∎

**Cách 4 (ngoại tuyến).** Bất biến: khi xử lý điểm `(x, y)` (theo `x` giảm dần), mảng `cnt` chứa đúng các hình chữ nhật có `l ≥ x`. Đúng vì: con trỏ `k` đã thêm mọi hình chữ nhật có `l ≥ x` (vòng while), và chưa thêm hình nào có `l < x` (danh sách sắp theo `l` giảm, vòng while dừng ở cái đầu tiên `< x`). Với các điểm sau có `x' ≤ x`, tập `l ≥ x'` là tập lớn hơn ⇒ chỉ cần **thêm**, không cần xóa. ∎

**Thứ tự đáp án.** Ta duyệt điểm theo thứ tự đã sắp nhưng ghi `ans[j]` với `j` là chỉ số gốc ⇒ đầu ra đúng thứ tự `points`.

## 11. Phân tích Time / Space Complexity

`R, P ≤ 5·10⁴`, `H = 100` (chiều cao tối đa).

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | O(R·P) | O(1) |
| 2. Nhóm + bisect | O(R log R + P · H · log R) | O(R) |
| 3. Gộp hậu tố theo h | O(H · R log R + P log R) | O(H · R) |
| 4. Ngoại tuyến x + mảng đếm/Fenwick | O((R + P) log + P · H) / O(P log H) | O(R + P) |
| 5. Ngoại tuyến y + SortedList | O((R + P) log R) | O(R + P) |

In [18]:
rng = random.Random(2250)
R = [[rng.randint(1, 10**9), rng.randint(1, 100)] for _ in range(5 * 10**4)]
P = [[rng.randint(1, 10**9), rng.randint(1, 100)] for _ in range(5 * 10**4)]
res = {}
for name, f in [("group_bisect", group_bisect), ("merged_suffix", merged_suffix), ("offline_by_x", offline_by_x),
                ("offline_fenwick", offline_fenwick), ("offline_by_y", offline_by_y), ("Solution", Solution().countRectangles)]:
    t = time.perf_counter(); res[name] = f(R, P); dt = time.perf_counter() - t
    print(f"{name:<16} {dt*1000:6.0f} ms")
assert all(v == res["Solution"] for v in res.values())

group_bisect        701 ms
merged_suffix       129 ms


offline_by_x         95 ms
offline_fenwick      98 ms


offline_by_y        157 ms


Solution            582 ms


## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [19]:
ALL = {"brute": brute, "group_bisect": group_bisect, "merged_suffix": merged_suffix, "offline_by_x": offline_by_x,
       "offline_fenwick": offline_fenwick, "offline_by_y": offline_by_y, "Solution": Solution().countRectangles}
edge = examples + [
    ([[1, 1]], [[1, 1]], [1]),                         # điểm đúng tại góc
    ([[1, 1]], [[2, 1], [1, 2]], [0, 0]),
    ([[10**9, 100]], [[10**9, 100], [1, 1]], [1, 1]),  # giá trị lớn nhất
    ([[5, 3], [5, 4], [6, 3]], [[5, 3]], [3]),         # trên cạnh
    ([[2, 2], [3, 1], [1, 3]], [[2, 2], [1, 1], [3, 3]], [1, 3, 0]),
]
for r, p, exp in edge:
    for name, f in ALL.items():
        rr = [list(x) for x in r]; pp = [list(x) for x in p]
        assert f(rr, pp) == exp, (name, r, p, f(r, p), exp)
        assert rr == r and pp == p
print("Edge case OK:", len(edge))

Edge case OK: 7


### 12.2 Đối chiếu ngẫu nhiên (seed cố định) — có nhiều điểm/hình cùng tọa độ x hoặc y

In [20]:
rng = random.Random(42)
for _ in range(500):
    lx = rng.choice([5, 20, 10**9]); hy = rng.choice([3, 10, 100])
    rs = list({(rng.randint(1, lx), rng.randint(1, hy)) for _ in range(rng.randint(1, 30))})
    ps = list({(rng.randint(1, lx), rng.randint(1, hy)) for _ in range(rng.randint(1, 30))})
    rs = [list(x) for x in rs]; ps = [list(x) for x in ps]
    ref = brute(rs, ps)
    for name, f in ALL.items():
        assert f(rs, ps) == ref, (name, rs, ps)
print("500 case ngẫu nhiên OK")

500 case ngẫu nhiên OK


### 12.3 Tính chất: tăng x hoặc y của điểm không làm tăng đáp án (đơn điệu)

In [21]:
rng = random.Random(7)
rs = [[rng.randint(1, 50), rng.randint(1, 20)] for _ in range(200)]
for _ in range(300):
    x, y = rng.randint(1, 50), rng.randint(1, 20)
    a = Solution().countRectangles(rs, [[x, y], [x + rng.randint(0, 5), y], [x, min(100, y + rng.randint(0, 5))]])
    assert a[1] <= a[0] and a[2] <= a[0]
print("tính chất đơn điệu OK")

tính chất đơn điệu OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dùng `bisect_right` (bỏ hình chữ nhật có cạnh đúng bằng x)

In [22]:
from bisect import bisect_right
def wrong_bisect_right(rectangles, points):
    by_h = defaultdict(list)
    for l, h in rectangles: by_h[h].append(l)
    for h in by_h: by_h[h].sort()
    return [sum(len(ls) - bisect_right(ls, x) for h, ls in by_h.items() if h >= y) for x, y in points]   # SAI

r, p = [[1, 1], [2, 2], [3, 3]], [[1, 3], [1, 1]]
print("bản sai :", wrong_bisect_right(r, p), "| bản đúng:", Solution().countRectangles(r, p))

bản sai : [1, 2] | bản đúng: [1, 3]


**Vì sao sai:** `bisect_right(ls, x)` đếm số phần tử `≤ x` ⇒ `len − bisect_right` là số `> x`, bỏ mất các hình chữ nhật có `l = x` (điểm nằm trên cạnh vẫn được tính). **Sửa:** `bisect_left`.

### Lỗi 2 — Chỉ xét nhóm có chiều cao **đúng bằng** y

In [23]:
def wrong_exact_height(rectangles, points):
    by_h = defaultdict(list)
    for l, h in rectangles: by_h[h].append(l)
    for h in by_h: by_h[h].sort()
    return [len(by_h.get(y, [])) - bisect_left(by_h.get(y, []), x) for x, y in points]   # SAI

r, p, _ = examples[0]
print("bản sai :", wrong_exact_height(r, p), "| bản đúng:", Solution().countRectangles(r, p))

bản sai : [0, 0] | bản đúng: [2, 1]


**Sửa:** cộng mọi nhóm `h ≥ y`.

### Lỗi 3 — Xử lý ngoại tuyến nhưng ghi đáp án theo thứ tự đã sắp

In [24]:
def wrong_offline_order(rectangles, points):
    rs = sorted(rectangles, key=lambda r: -r[0]); ps = sorted(points, key=lambda p: -p[0])
    cnt = [0] * 102; ans = []; k = 0
    for x, y in ps:
        while k < len(rs) and rs[k][0] >= x:
            cnt[rs[k][1]] += 1; k += 1
        ans.append(sum(cnt[y:101]))                     # SAI: thứ tự của ps, không phải points
    return ans

r, p = [[1, 1], [5, 5]], [[1, 1], [5, 5]]
print("bản sai :", wrong_offline_order(r, p), "| bản đúng:", Solution().countRectangles(r, p))

bản sai : [1, 2] | bản đúng: [2, 1]


**Sửa:** sắp xếp **chỉ số** (`order = sorted(range(P), key=…)`) và ghi `ans[j]`.

### Lỗi 4 — Mảng đếm theo chiều dài (10⁹ phần tử)

In [25]:
def wrong_length_array(rectangles, points):
    L = max(l for l, _ in rectangles)
    if L > 10**7:
        return f"MemoryError (mô phỏng): cần mảng {L} phần tử cho mỗi chiều cao"
    return None

print("bản sai :", wrong_length_array([[10**9, 1]], [[1, 1]]))
print("bản đúng:", Solution().countRectangles([[10**9, 1]], [[1, 1]]))

bản sai : MemoryError (mô phỏng): cần mảng 1000000000 phần tử cho mỗi chiều cao
bản đúng: [1]


**Vì sao sai:** chiều dài tới 10⁹ ⇒ không thể tạo mảng/tiền tố theo chiều dài. Chỉ chiều **cao** (≤ 100) mới dùng mảng được. **Sửa:** đánh chỉ số theo chiều cao, tìm nhị phân theo chiều dài.

### Lỗi 5 — `insort` vào list thường trong cách 5 (O(n) mỗi lần chèn)

In [26]:
def offline_by_y_insort(rectangles, points):
    rs = sorted(rectangles, key=lambda r: -r[1]); order = sorted(range(len(points)), key=lambda j: -points[j][1])
    lst = []; ans = [0] * len(points); k = 0
    for j in order:
        x, y = points[j]
        while k < len(rs) and rs[k][1] >= y:
            insort(lst, rs[k][0]); k += 1                    # O(n) dịch phần tử
        ans[j] = len(lst) - bisect_left(lst, x)
    return ans

rng = random.Random(5)
R2 = [[rng.randint(1, 10**9), rng.randint(1, 100)] for _ in range(20000)]
P2 = [[rng.randint(1, 10**9), rng.randint(1, 100)] for _ in range(20000)]
t = time.perf_counter(); a = offline_by_y_insort(R2, P2); t1 = time.perf_counter() - t
t = time.perf_counter(); b = offline_by_y(R2, P2); t2 = time.perf_counter() - t
print(f"cùng kết quả: {a == b}; insort {t1*1000:.0f} ms vs SortedList {t2*1000:.0f} ms")

cùng kết quả: True; insort 36 ms vs SortedList 35 ms


Với 2·10⁴ phần tử `insort` vẫn nhanh nhờ `memmove` trong C, nhưng độ phức tạp là O(R²) — với dữ liệu lớn hơn (10⁶) sẽ chậm rõ rệt. **Sửa:** `SortedList` (O(log n) mỗi lần thêm).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **Đọc ràng buộc từng chiều**: một chiều nhỏ (≤ 100) ⇒ duyệt/đánh chỉ số theo chiều đó, chiều lớn ⇒ sắp xếp + tìm nhị phân.
- "Đếm số cặp `(a, b)` với `a ≥ x` và `b ≥ y`" ⇒ **dominance counting**: ngoại tuyến (sắp theo một chiều, Fenwick/SortedList theo chiều kia).
- Ngoại tuyến: sắp **chỉ số** truy vấn, ghi đáp án theo chỉ số gốc.
- `bisect_left` cho "≥ x", `bisect_right` cho "> x".
- Bài cùng họ: 1478, 315 Count of Smaller Numbers After Self (Fenwick), 2251 Number of Flowers in Full Bloom, 1847 Closest Room (ngoại tuyến + SortedList), 2070.

## 15. Biến thể của bài

### 15.1 Biến thể A — Hình chữ nhật **tổng quát** `[x1, y1, x2, y2]` (không gắn gốc), ít dữ liệu

Không còn tính "trội" một phía; với dữ liệu nhỏ, vét cạn có điều kiện bốn bất đẳng thức. (Bản lớn cần quét đường + cây đoạn.)

In [27]:
def count_general(rects, points):
    return [sum(1 for x1, y1, x2, y2 in rects if x1 <= x <= x2 and y1 <= y <= y2) for x, y in points]
print(count_general([[0, 0, 2, 2], [1, 1, 3, 3], [2, 0, 4, 1]], [[2, 1], [0, 3], [3, 3]]))

[3, 0, 1]


### 15.2 Biến thể B — Cả hai chiều tới 10⁹ (dominance counting tổng quát)

Cách 5 (ngoại tuyến theo y + `SortedList`) không phụ thuộc chiều cao nhỏ. Kiểm chứng:

In [28]:
rng = random.Random(152)
for _ in range(200):
    rs = [[rng.randint(1, 10**9), rng.randint(1, 10**9)] for _ in range(rng.randint(1, 40))]
    ps = [[rng.randint(1, 10**9), rng.randint(1, 10**9)] for _ in range(rng.randint(1, 40))]
    assert offline_by_y(rs, ps) == brute(rs, ps)
print("dominance counting tổng quát: khớp brute force")

dominance counting tổng quát: khớp brute force


### 15.3 Biến thể C — Mỗi hình chữ nhật có **trọng số** `w`; trả về tổng trọng số chứa điểm

Thay "thêm 1" bằng "thêm `w`" trong Fenwick (cách 4).

In [29]:
def weighted(rects, points):
    rs = sorted(rects, key=lambda r: -r[0]); order = sorted(range(len(points)), key=lambda j: -points[j][0])
    fw = Fenwick(100); ans = [0] * len(points); k = 0; total = 0
    for j in order:
        x, y = points[j]
        while k < len(rs) and rs[k][0] >= x:
            fw.add(rs[k][1], rs[k][2]); total += rs[k][2]; k += 1
        ans[j] = total - fw.prefix(y - 1)
    return ans

rw = [[1, 2, 5], [2, 3, 7], [2, 5, 1]]
print(weighted(rw, [[2, 1], [1, 4]]), "| brute:", [sum(w for l, h, w in rw if x <= l and y <= h) for x, y in [[2, 1], [1, 4]]])

[8, 1] | brute: [8, 1]


### 15.4 Biến thể D — Đếm **điểm** nằm trong mỗi hình chữ nhật (đảo vai)

Với mỗi hình chữ nhật `(l, h)`: số điểm có `x ≤ l` và `y ≤ h`. Nhóm điểm theo `y` (≤ 100), mỗi nhóm sắp `x`, dùng `bisect_right`.

In [30]:
def points_in_each_rect(rectangles, points):
    by_y = defaultdict(list)
    for x, y in points: by_y[y].append(x)
    for y in by_y: by_y[y].sort()
    return [sum(bisect_right(xs, l) for y, xs in by_y.items() if y <= h) for l, h in rectangles]

r, p, _ = examples[0]
print(points_in_each_rect(r, p), "| brute:", [sum(1 for x, y in p if x <= l and y <= h) for l, h in r])

[0, 1, 2] | brute: [0, 1, 2]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | mọi cặp | O(R·P) | O(1) | kiểm chứng |
| 2. Nhóm chiều cao + bisect | H nhỏ, mỗi nhóm tìm nhị phân | O(R log R + P·H log R) | O(R) | **mặc định** (khai thác H ≤ 100) |
| 3. Gộp hậu tố theo h | một bisect mỗi truy vấn | O(H·R log R + P log R) | O(H·R) | rất nhiều truy vấn, dư bộ nhớ |
| 4. Ngoại tuyến x + đếm/Fenwick | quét x, đếm theo h | O((R+P) log + P log H) | O(R+P) | thêm trọng số, cập nhật |
| 5. Ngoại tuyến y + SortedList | quét y, tìm nhị phân theo l | O((R+P) log R) | O(R+P) | cả hai chiều lớn |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `countRectangles([[4,7],[4,9],[8,2],[5,2],[3,4]], [[1,4],[4,2],[5,8],[9,9]])`?

**Bài 2 (tính tay):** Với `rectangles = [[3,1],[3,2],[1,2]]`, điểm `(2, 2)` và `(3, 1)` được chứa bởi bao nhiêu hình?

**Bài 3 (điền TODO):** Viết `bt3_todo(rectangles, points)` trả về với mỗi điểm **danh sách chỉ số** hình chữ nhật chứa nó (thứ tự tăng), dùng nhóm theo chiều cao (lưu cặp `(l, i)`).

**Bài 4 (tự viết lại):** Viết `bt4_todo(rectangles, points)` cài cách 4 (ngoại tuyến theo x + mảng đếm) không nhìn code.

**Bài 5 (mini challenge):** Viết `bt5_todo(rectangles, points)` trả về số **cặp (điểm, hình chữ nhật)** mà hình chứa điểm, với cả hai chiều tới 10⁹, trong O((R + P) log) — không liệt kê cặp.

In [31]:
def bt3_todo(rectangles, points):
    # TODO: danh sách chỉ số hình chứa mỗi điểm
    pass

def bt4_todo(rectangles, points):
    # TODO: ngoại tuyến theo x + mảng đếm theo chiều cao
    pass

def bt5_todo(rectangles, points):
    # TODO: tổng số cặp chứa, cả hai chiều lớn
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: điểm `(1, 4)`: cần `l ≥ 1, h ≥ 4`. Mức 2: `[4,7],[4,9],[3,4]` ⇒ 3. Mức 3: đáp án `[3, 4, 0, 0]` (điểm `(4, 2)`: `[4,7],[4,9],[8,2],[5,2]`).

**Bài 2.** Mức 1: `(2, 2)` cần `l ≥ 2, h ≥ 2`. Mức 2: chỉ `[3,2]`. Mức 3: `(2,2) → 1`; `(3,1) → 2` (`[3,1], [3,2]`).

**Bài 3.** Mức 1: `by_h[h]` chứa `(l, i)` đã sắp. Mức 2: `bisect_left(ls, (x, -1))` để lấy mọi `l ≥ x`. Mức 3: gom chỉ số các phần tử từ vị trí đó, sắp tăng.

**Bài 4.** Mức 1: sắp hình theo `l` giảm, chỉ số điểm theo `x` giảm. Mức 2: con trỏ thêm `cnt[h] += 1`. Mức 3: `ans[j] = sum(cnt[y:101])`.

**Bài 5.** Mức 1: tổng = Σ_j count[j]. Mức 2: dùng cách 5 (SortedList) rồi cộng. Mức 3: hoặc nén tọa độ y và Fenwick — cùng O((R + P) log).

## 19. Đáp án tham khảo cho bài tập

**Bài 1:** `[3, 4, 0, 0]`. **Bài 2:** `1`, `2`.

In [32]:
assert Solution().countRectangles([[4, 7], [4, 9], [8, 2], [5, 2], [3, 4]], [[1, 4], [4, 2], [5, 8], [9, 9]]) == [3, 4, 0, 0]
assert Solution().countRectangles([[3, 1], [3, 2], [1, 2]], [[2, 2], [3, 1]]) == [1, 2]

def bt3(rectangles, points):
    by_h = defaultdict(list)
    for i, (l, h) in enumerate(rectangles): by_h[h].append((l, i))
    for h in by_h: by_h[h].sort()
    out = []
    for x, y in points:
        ids = []
        for h, ls in by_h.items():
            if h >= y:
                ids += [i for _, i in ls[bisect_left(ls, (x, -1)):]]
        out.append(sorted(ids))
    return out

def bt4(rectangles, points):
    rs = sorted(rectangles, key=lambda r: -r[0]); order = sorted(range(len(points)), key=lambda j: -points[j][0])
    cnt = [0] * 102; ans = [0] * len(points); k = 0
    for j in order:
        x, y = points[j]
        while k < len(rs) and rs[k][0] >= x:
            cnt[rs[k][1]] += 1; k += 1
        ans[j] = sum(cnt[y:101])
    return ans

def bt5(rectangles, points):
    return sum(offline_by_y(rectangles, points))

rng = random.Random(1950)
for _ in range(400):
    rs = [list(x) for x in {(rng.randint(1, 20), rng.randint(1, 10)) for _ in range(rng.randint(1, 25))}]
    ps = [list(x) for x in {(rng.randint(1, 20), rng.randint(1, 10)) for _ in range(rng.randint(1, 25))}]
    ref = brute(rs, ps)
    assert [len(v) for v in bt3(rs, ps)] == ref
    assert all(all(ps[j][0] <= rs[i][0] and ps[j][1] <= rs[i][1] for i in ids) for j, ids in enumerate(bt3(rs, ps)))
    assert bt4(rs, ps) == ref
    big_r = [[rng.randint(1, 10**9), rng.randint(1, 10**9)] for _ in range(20)]
    big_p = [[rng.randint(1, 10**9), rng.randint(1, 10**9)] for _ in range(20)]
    assert bt5(big_r, big_p) == sum(brute(big_r, big_p))
print("Đáp án bài tập: tất cả assert OK")

Đáp án bài tập: tất cả assert OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
by_h = defaultdict(list)                       # chiều nhỏ làm khóa
for l, h in rectangles: by_h[h].append(l)
heights = sorted(by_h)
for h in heights: by_h[h].sort()
for x, y in points:
    c = 0
    for h in heights[bisect_left(heights, y):]:
        ls = by_h[h]; c += len(ls) - bisect_left(ls, x)   # số l >= x
    ans.append(c)
```

**Checklist:**
- [ ] Khai thác chiều cao ≤ 100; **không** đánh chỉ số theo chiều dài (10⁹).
- [ ] `bisect_left` cho "≥ x" (điểm trên cạnh vẫn tính).
- [ ] Cộng **mọi** chiều cao `≥ y`.
- [ ] Ngoại tuyến: ghi đáp án theo **chỉ số gốc**.
- [ ] Đã thử: điểm trùng góc, điểm trên cạnh, giá trị 10⁹.